# Tahap 2 — Case Representation (v3 — Deep Clean)

**Input**: `data/processed/clean/pasal_362/*.txt` dan `data/processed/clean/pasal_363/*.txt`

**Output**: `data/processed/cases.csv` dan `data/processed/cases.json`

### Perbaikan v3
1. Preprocessing line-by-line: hapus disclaimer, footer halaman, nomor halaman per baris
2. Teks penuh (bukan `text[:3000]`)
3. Word splitting dictionary-based untuk kata menyatu akibat line break PDF
4. Two-pass: bangun kamus kata dari pass pertama, gunakan untuk split di pass kedua

## 2.1 Import Library

In [1]:
import re
import json
import logging
import pandas as pd
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm

In [2]:
CLEAN_DIR  = Path("../data/processed/clean")
OUTPUT_DIR = Path("../data/processed")
LOG_DIR    = Path("../logs")

for d in [OUTPUT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "representation.log", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger(__name__)

## 2.2 Fungsi Pembersihan Teks (Deep Clean)

### Struktur file mentah per halaman:
```
[konten halaman — satu baris panjang dengan kata menyatu]
putusan nomor XXX/pid.b/YYYY/pn mlg          ← footer halaman
[baris kosong x5-7]
disclaimer                                     ← blok disclaimer MA
pelaksanaan fungsi peradilan...                
email : kepaniteraan@mahkamahagung.go.id ...(ext.318)
[baris kosong]
halaman N                                      ← nomor halaman
[baris kosong]
[konten halaman berikutnya]
```

### Pipeline pembersihan:
1. **Line-by-line**: hapus baris disclaimer, halaman N, baris kosong
2. **Strip footer** dari setiap baris konten
3. **Gabung** baris konten dengan spasi
4. **Split kata menyatu** menggunakan kamus kata

In [3]:
def remove_page_boundaries(raw_text):
    """
    Proses line-by-line:
    - Hapus blok disclaimer (dari 'disclaimer' sampai 'ext.318)')
    - Hapus 'halaman N'
    - Hapus baris kosong
    - Strip footer 'putusan nomor .../pn mlg' dari akhir baris konten
    """
    lines = raw_text.split('\n')
    content_lines = []
    in_disclaimer = False

    for line in lines:
        stripped = line.strip()

        if not stripped:
            continue

        if stripped.lower() == 'disclaimer':
            in_disclaimer = True
            continue

        if in_disclaimer:
            if 'ext.318)' in stripped.lower():
                in_disclaimer = False
            continue

        if re.match(r'^halaman\s+\d+$', stripped, re.IGNORECASE):
            continue

        stripped = re.sub(
            r'\s*putusan\s+nomor\s+\d+/pid\.\w+/\d{4}/pn\s+\w+\s*$',
            '',
            stripped,
            flags=re.IGNORECASE
        )

        if stripped:
            content_lines.append(stripped)

    return ' '.join(content_lines)

In [4]:
INDO_COMMON_WORDS = {
    "ada", "adalah", "adanya", "agar", "akan", "akibat", "akhir",
    "alasan", "amar", "anak", "antara", "apa", "apabila",
    "atas", "atau", "ayat",
    "bagi", "bagian", "bahwa", "baik", "barang", "baru",
    "batu", "beberapa", "belum", "benar", "berapa",
    "berdasarkan", "berdiri", "berikut", "berkas", "bermaksud",
    "bersalah", "bersama", "berupa", "besar", "biasa", "biaya",
    "bila", "bisa", "boleh", "buah", "bukan", "bulan",
    "cara", "cukup",
    "dahulu", "dalam", "dan", "dapat", "dari", "datang",
    "dengan", "demi", "demikian", "demikianlah",
    "denda", "depan", "desa", "dimana", "dimaksud",
    "diatur", "diancam", "didakwa", "didakwakan", "dijatuhkan",
    "dinyatakan", "diperoleh", "dipersidangan",
    "duduk", "dusun",
    "empat", "enam",
    "fakta", "foto",
    "ganti",
    "hak", "hal", "halaman", "hanya", "hari", "hakim",
    "harus", "haruslah", "hasil", "hendak", "hukum", "hukuman",
    "ialah", "ini", "islam", "itu",
    "jadi", "jalan", "jaksa", "jam", "jenis",
    "jika", "juga", "jumlah",
    "kabupaten", "kalau", "kami", "karena", "karenanya",
    "kasus", "kata", "keadilan", "kebangsaan",
    "kecamatan", "kedua", "kegiatan", "kelamin",
    "kelurahan", "keluarga", "kemudian", "kepada",
    "kepadanya", "kepentingan", "keperluan",
    "kerja", "kerugian", "kesadaran",
    "keterangan", "ketentuan", "ketiga", "ketika",
    "ketua", "ketuhanan",
    "kewajiban", "khususnya",
    "korban", "kota", "kuasa", "kunci", "kurang",
    "lagi", "lahir", "lain", "lalu", "lama", "lampiran",
    "langsung", "lebih", "lengkap",
    "lewat", "lima",
    "maha", "maka", "maksud", "malang",
    "mampu", "mana", "masa", "masing",
    "masih", "masuk", "masyarakat", "maupun",
    "majelis", "meja",
    "melakukan", "melanggar", "melawan",
    "melepaskan", "memang", "membebaskan",
    "membawa", "membayar", "memberi", "memberikan",
    "memenuhi", "memiliki",
    "memohon", "memotong", "mempertimbangkan",
    "memperhatikan",
    "memutuskan", "menarik",
    "mendapat", "mendekati", "mendorong",
    "mendengar", "menerangkan",
    "mengadili", "mengajukan", "mengakui", "mengalami",
    "mengambil", "mengenai", "mengetahui",
    "menggunakan", "mengingat",
    "menguasai",
    "menimbang", "menikmati", "menjadi",
    "menjatuhkan", "menuju",
    "menurut", "menyatakan", "menyesal", "menyesali",
    "menyebutkan",
    "meresahkan", "merupakan", "meskipun",
    "milik", "miliki", "miliknya", "motor", "mulai",
    "nama", "namun", "negeri", "nomor",
    "orang", "oleh",
    "pada", "padahal", "pagi", "paling",
    "panitera", "para", "parkir",
    "pasal", "pasti", "pekerjaan",
    "pelaku", "pemaaf", "pembenar",
    "pembacaan", "pemeriksaan",
    "pencurian", "pendapat",
    "penetapan",
    "pengganti", "pengetahuan",
    "penilaian",
    "penjara", "penjeraan",
    "penuntut", "penunjukan",
    "penyesuaian", "penyidik",
    "peradilan", "peraturan",
    "perbuatan", "perbuatannya",
    "perkara",
    "perlu", "pernah", "persidangan",
    "pertama", "pertimbangan",
    "perundang",
    "pidana", "pihak", "pokoknya",
    "pula", "pukul",
    "putusan",
    "rasa", "rekaman", "republik", "resto",
    "ringan", "rupiah",
    "saat", "saja", "saksi", "salah", "saling",
    "sama", "sampai", "sangat", "sarana",
    "satu", "sebagai", "sebagaimana",
    "sebagian", "sebelum", "sebelumnya",
    "secara", "sedang", "sedangkan",
    "sehari", "sehingga",
    "sejak", "sejumlah", "sekira", "sekitar",
    "selain", "selaku", "selama", "selanjutnya",
    "seluruh", "seluruhnya",
    "semua", "sendiri",
    "sependapat",
    "sepeda", "sepengetahuan",
    "seperti", "sepatutnya",
    "serta", "sesuai", "sesuatu",
    "setelah", "setiap", "setimpal",
    "sifat", "singkat", "sisi", "sopan",
    "suatu", "sudah", "sumpah", "supaya", "surat",
    "tahun", "tanggal", "tanpa", "telah",
    "tempat", "tentang", "tentunya",
    "terbukti", "terhadap",
    "terdakwa", "terlebih", "termasuk", "ternyata",
    "tersebut", "terungkap",
    "tetap", "tetapi",
    "tidak", "tindak", "tinggal",
    "tujuan", "tulang", "tuntutan",
    "tujuh", "tiga", "tingkat",
    "umum", "umur", "undang",
    "unsur", "untuk", "utama",
    "wajib", "waktu", "walaupun", "warna", "wib",
}

In [5]:
def build_word_set_from_corpus(texts, base_words=None):
    if base_words is None:
        base_words = set()

    word_freq = Counter()
    for text in texts:
        for token in text.lower().split():
            word = re.sub(r'[^a-z]', '', token)
            if 3 <= len(word) <= 20:
                word_freq[word] += 1

    corpus_words = {w for w, c in word_freq.items() if c >= 3 and 3 <= len(w) <= 18}

    return base_words | corpus_words

In [6]:
def split_merged_words(text, word_set, min_token_len=6, min_part_len=3):
    tokens = text.split()
    result = []

    for token in tokens:
        alpha_only = re.sub(r'[^a-z]', '', token)

        if len(alpha_only) <= min_token_len or alpha_only in word_set:
            result.append(token)
            continue

        best_split = None
        best_score = -1

        for i in range(min_part_len, len(alpha_only) - min_part_len + 1):
            left = alpha_only[:i]
            right = alpha_only[i:]

            left_known = left in word_set
            right_known = right in word_set

            if left_known and right_known:
                score = len(left) + len(right) + min(len(left), len(right))
                if score > best_score:
                    best_score = score
                    best_split = (left, right)

        if best_split:
            result.append(best_split[0])
            remaining = best_split[1]
            sub_result = split_merged_words(remaining, word_set, min_token_len, min_part_len)
            result.append(sub_result)
        else:
            result.append(token)

    return ' '.join(result)

In [7]:
def deep_clean_text(raw_text, word_set):
    text = remove_page_boundaries(raw_text)
    text = text.lower()

    def fix_spaced_chars(m):
        return m.group(0).replace(' ', '')
    text = re.sub(r'(?<!\w)([a-z] ){3,}[a-z](?!\w)', fix_spaced_chars, text)

    text = re.sub(r';', '; ', text)
    text = re.sub(r'(?<=[a-z])\.(?=[a-z])', '. ', text)

    text = re.sub(r'\s+', ' ', text).strip()

    text = split_merged_words(text, word_set)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

## 2.3 Pass 1: Pembersihan Dasar + Bangun Kamus Kata

In [8]:
txt_files = sorted(CLEAN_DIR.rglob("*.txt"))
print(f"Total file ditemukan: {len(txt_files)}")

pass1_texts = []
for txt_path in tqdm(txt_files, desc="Pass 1: Pembersihan dasar"):
    raw = txt_path.read_text(encoding="utf-8", errors="ignore")
    cleaned = remove_page_boundaries(raw).lower()
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    pass1_texts.append(cleaned)

word_set = build_word_set_from_corpus(pass1_texts, INDO_COMMON_WORDS)
print(f"Kamus kata: {len(word_set)} kata unik")
print(f"Contoh: {sorted(list(word_set))[:20]}")

Total file ditemukan: 60


Pass 1: Pembersihan dasar:   0%|          | 0/60 [00:00<?, ?it/s]

Kamus kata: 6624 kata unik
Contoh: ['aap', 'aaptahun', 'aasj', 'abdul', 'abdulaziz', 'abdullah', 'abdurrahman', 'abdurrohman', 'abi', 'abidin', 'abn', 'abq', 'abr', 'absenuntuk', 'abu', 'abuabu', 'abumetalik', 'abyss', 'abyssblue', 'acak']


## 2.4 Pass 2: Deep Clean + Split Kata Menyatu

In [9]:
sample_raw = txt_files[0].read_text(encoding="utf-8", errors="ignore")
sample_cleaned = deep_clean_text(sample_raw, word_set)

print("=== SEBELUM (500 char pertama dari raw) ===")
print(sample_raw[:500])
print()
print("=== SESUDAH (500 char pertama dari cleaned) ===")
print(sample_cleaned[:500])
print()
print(f"Raw: {len(sample_raw)} chars → Clean: {len(sample_cleaned)} chars")
print(f"Noise dihapus: {len(sample_raw) - len(sample_cleaned)} chars ({(1 - len(sample_cleaned)/len(sample_raw))*100:.1f}%)")

=== SEBELUM (500 char pertama dari raw) ===
disclaimer
 
pelaksanaan fungsi peradilan. namun dalam hal-hal tertentu masih dimungkinkan terjadi permasalahan teknis terkait dengan akurasi dan keterkinian informasi yang kami sajikan, hal mana akan terus kami perbaiki dari waktu kewaktu.
dalam hal anda menemukan inakurasi informasi yang termuat pada situs ini atau informasi yang seharusnya ada, namun belum tersedia, maka harap segera hubungi 
email : kepaniteraan@mahkamahagung.go.id telp : 021-384 3348 (ext.318)

halaman 1

putusannomor 132/p

=== SESUDAH (500 char pertama dari cleaned) ===
putusan nomor 132/pid. b/2026/pn mlgdemi keadilan berdasarkan ketuhanan yang maha esapengadilan negeri malang yang mengadili perkara pidana denganacara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagaiberikut dalam perkara terdakwa :1.nama lengkap: sendi sulistiyadi 2.tempat lahir: bandung umurtanggal lahir: 39 tahun / 21 september 19864.jenis kelamin: laki-laki ; 5.kebangsaan: indones

In [10]:
long_tokens_before = [t for t in pass1_texts[0].split() if len(t) > 15 and t.isalpha()]
long_tokens_after  = [t for t in sample_cleaned.split() if len(t) > 15 and t.isalpha()]

print(f"Token panjang (>15 char) SEBELUM split: {len(long_tokens_before)}")
for t in long_tokens_before[:10]:
    print(f"  - {t}")

print(f"\nToken panjang (>15 char) SESUDAH split: {len(long_tokens_after)}")
for t in long_tokens_after[:10]:
    print(f"  - {t}")

Token panjang (>15 char) SEBELUM split: 59
  - sebagaimanadalam
  - sulistiyadiberupa
  - menyesaliperbuatannya
  - lisandipersidangan
  - pokoknyamenyatakan
  - kecamatanlowokwaru
  - akandigunakannya
  - menyalakansepeda
  - mengalamikerugian
  - pokoknyamenerangkan

Token panjang (>15 char) SESUDAH split: 41
  - sebagaimanadalam
  - pokoknyamenyatakan
  - kecamatanlowokwaru
  - akandigunakannya
  - menyalakansepeda
  - denganmengendarai
  - mengalamikerugian
  - tanpasepengetahuan
  - terdakwakemudian
  - mempertimbangkan


## 2.5 Fungsi Ekstraksi Metadata

In [11]:
def ekstrak_no_perkara(text):
    patterns = [
        r"\d+\s*/\s*pid\.b\s*/\s*\d{4}\s*/\s*pn\s*[\w\.]+",
        r"\d+\s*/\s*pid\.sus\s*/\s*\d{4}\s*/\s*pn\s*[\w\.]+",
        r"nomor\s*[:\s]+(\d+[/\-][\w\.]+[/\-]\d{4}[/\-][\w\.]+)",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return m.group(0).strip()
    return ""


def ekstrak_tanggal(text):
    bulan_map = {
        "januari": "01", "februari": "02", "maret": "03", "april": "04",
        "mei": "05", "juni": "06", "juli": "07", "agustus": "08",
        "september": "09", "oktober": "10", "november": "11", "desember": "12"
    }
    pat = r"(\d{1,2})\s+(januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember)\s+(\d{4})"
    m = re.search(pat, text, re.IGNORECASE)
    if m:
        hari, bln, thn = m.group(1), m.group(2).lower(), m.group(3)
        return f"{thn}-{bulan_map.get(bln, '00')}-{int(hari):02d}"
    return ""


def ekstrak_terdakwa(text):
    patterns = [
        r"terdakwa\s*[:\s]+([A-Z][a-zA-Z\s]+?)(?:\s*,|\s*alias|\s*bin|\s*binti|\n)",
        r"nama\s*lengkap\s*[:\s]+([A-Z][a-zA-Z\s]+?)(?:\s*;|\s*,|\n)",
        r"nama\s*[:\s]+([A-Z][a-zA-Z\s\.]+?)(?:\s*;|\s*,|\s*\n)",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            nama = m.group(1).strip()
            if 3 < len(nama) < 60:
                return nama.title()
    return ""


def ekstrak_pasal_dakwaan(text):
    patterns = [
        r"pasal\s+362\s+kuhp",
        r"pasal\s+363\s+(?:ayat\s+\(\d\)\s+)?kuhp",
        r"pasal\s+364\s+kuhp",
        r"pasal\s+365\s+kuhp",
        r"pasal\s+476\b",
        r"pasal\s+3[56789]\d?\s+(?:jo\.?\s+pasal\s+\d+\s+)?kuhp",
    ]
    found = []
    for pat in patterns:
        matches = re.findall(pat, text, re.IGNORECASE)
        found.extend(matches)
    return "; ".join(set(found)) if found else ""


def ekstrak_amar_putusan(text):
    m = re.search(
        r"m\s*e\s*n\s*g\s*a\s*d\s*i\s*l\s*i(.{50,800}?)(?=menimbang|menetapkan|demikian|$)",
        text, re.IGNORECASE | re.DOTALL
    )
    if m:
        blok = m.group(1).lower().strip()
        if re.search(r"terbukti.*bersalah|menyatakan.*terdakwa.*bersalah", blok):
            return "bersalah"
        if re.search(r"tidak terbukti|membebaskan terdakwa", blok):
            return "bebas"
        if re.search(r"melepaskan terdakwa|lepas dari", blok):
            return "lepas"
        return blok[:200].replace("\n", " ").strip()
    return ""


def ekstrak_vonis_bulan(text):
    tahun_m = re.search(r"pidana penjara selama\s+(\d+)\s*\([\w\s]+\)\s*tahun", text, re.IGNORECASE)
    bulan_m = re.search(r"pidana penjara selama\s+(\d+)\s*\([\w\s]+\)\s*bulan", text, re.IGNORECASE)
    if not tahun_m and not bulan_m:
        tahun_m = re.search(r"penjara\s+(\d+)\s+tahun", text, re.IGNORECASE)
        bulan_m = re.search(r"penjara\s+(\d+)\s+bulan", text, re.IGNORECASE)
    total = 0
    if tahun_m:
        total += int(tahun_m.group(1)) * 12
    if bulan_m:
        total += int(bulan_m.group(1))
    return total


def ekstrak_ringkasan_fakta(text):
    patterns = [
        r"menimbang.*?bahwa(.{100,800}?)(?=menimbang|mempertimbangkan|mengenai|$)",
        r"bahwa terdakwa(.{100,600}?)(?=bahwa|menimbang|$)",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE | re.DOTALL)
        if m:
            snippet = re.sub(r"\s+", " ", m.group(1).replace("\n", " ").strip())
            return snippet[:600]
    return ""


def ekstrak_argumen_hukum(text):
    patterns = [
        r"mempertimbangkan(.{100,800}?)(?=menimbang|mengadili|menetapkan|$)",
        r"pertimbangan hukum(.{100,600}?)(?=mengadili|menetapkan|$)",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE | re.DOTALL)
        if m:
            snippet = re.sub(r"\s+", " ", m.group(1).replace("\n", " ").strip())
            return snippet[:600]
    return ""

## 2.6 Proses Ekstraksi dengan Deep Clean

In [12]:
rows = []

for txt_path in tqdm(txt_files, desc="Pass 2: Deep clean + ekstraksi"):
    label_folder = txt_path.parent.name
    raw_text = txt_path.read_text(encoding="utf-8", errors="ignore")
    cleaned = deep_clean_text(raw_text, word_set)

    row = {
        "case_id":         txt_path.stem,
        "filename":        txt_path.name,
        "label_pasal":     label_folder,
        "no_perkara":      ekstrak_no_perkara(cleaned),
        "tanggal":         ekstrak_tanggal(cleaned),
        "pengadilan":      "PN Malang",
        "terdakwa":        ekstrak_terdakwa(cleaned),
        "pasal_dakwaan":   ekstrak_pasal_dakwaan(cleaned),
        "amar_putusan":    ekstrak_amar_putusan(cleaned),
        "vonis_bulan":     ekstrak_vonis_bulan(cleaned),
        "ringkasan_fakta": ekstrak_ringkasan_fakta(cleaned),
        "argumen_hukum":   ekstrak_argumen_hukum(cleaned),
        "word_count":      len(cleaned.split()),
        "text_full":       cleaned,
    }
    rows.append(row)

print(f"Total kasus diekstrak: {len(rows)}")

Pass 2: Deep clean + ekstraksi:   0%|          | 0/60 [00:00<?, ?it/s]

Total kasus diekstrak: 60


## 2.7 Simpan ke CSV dan JSON

In [13]:
df = pd.DataFrame(rows)

csv_path = OUTPUT_DIR / "cases.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"CSV disimpan: {csv_path} ({len(df)} baris)")

json_path = OUTPUT_DIR / "cases.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print(f"JSON disimpan: {json_path}")

CSV disimpan: ..\data\processed\cases.csv (60 baris)
JSON disimpan: ..\data\processed\cases.json


## 2.8 Statistik dan Validasi

In [14]:
print("Distribusi label_pasal:")
print(df["label_pasal"].value_counts().to_string())

print(f"\nDistribusi vonis_bulan:")
print(df["vonis_bulan"].describe().to_string())

vonis_zero = (df["vonis_bulan"] == 0).sum()
print(f"\nVonis 0 bulan (tidak terdeteksi): {vonis_zero}")
print(f"Vonis terdeteksi: {len(df) - vonis_zero}")

text_lengths = df["text_full"].str.len()
print(f"\nPanjang text_full:")
print(f"  Min: {text_lengths.min():,} | Max: {text_lengths.max():,} | Avg: {text_lengths.mean():,.0f}")
print(f"  Rata-rata word count: {df['word_count'].mean():.0f} kata")

all_long = []
for t in df['text_full']:
    for tok in t.split():
        if len(tok) > 15 and tok.isalpha():
            all_long.append(tok)
print(f"\nSisa token panjang >15 char di seluruh corpus: {len(all_long)}")
if all_long:
    print("Contoh:")
    for t in Counter(all_long).most_common(10):
        print(f"  {t[0]} ({t[1]}x)")

preview_cols = ["case_id", "label_pasal", "no_perkara", "amar_putusan", "vonis_bulan", "word_count"]
df[preview_cols].head(10)

Distribusi label_pasal:
label_pasal
pasal_362    30
pasal_363    30

Distribusi vonis_bulan:
count    60.000000
mean     14.433333
std      16.167939
min       0.000000
25%       0.000000
50%      12.000000
75%      24.000000
max      72.000000

Vonis 0 bulan (tidak terdeteksi): 20
Vonis terdeteksi: 40

Panjang text_full:
  Min: 21,050 | Max: 125,621 | Avg: 43,472
  Rata-rata word count: 5949 kata

Sisa token panjang >15 char di seluruh corpus: 3000
Contoh:
  mempertimbangkan (206x)
  pertanggungjawaban (122x)
  kemudianterdakwa (70x)
  menerangkansebagai (53x)
  penahananterhadap (51x)
  denganmenggunakan (51x)
  sertamemperhatikan (49x)
  melakukanpencurian (45x)
  putrinelghiviana (44x)
  pengadilannegeri (40x)


,case_id,label_pasal,no_perkara,amar_putusan,vonis_bulan,word_count
0,case_pasal_362_001,pasal_362,,bersalah,12,3760
1,case_pasal_362_002,pasal_362,,bersalah,12,5228
2,case_pasal_362_003,pasal_362,,bersalah,24,7243
3,case_pasal_362_004,pasal_362,,bersalah,12,2851
4,case_pasal_362_005,pasal_362,,bersalah,24,7593
5,case_pasal_362_006,pasal_362,,bersalah,72,17785
6,case_pasal_362_007,pasal_362,,bersalah,5,6238
7,case_pasal_362_008,pasal_362,,bersalah,10,3783
8,case_pasal_362_009,pasal_362,,bersalah,12,3994
9,case_pasal_362_010,pasal_362,,bersalah,12,4612


In [15]:
print("=== PREVIEW TEKS BERSIH (kasus pertama, 800 char) ===")
print(rows[0]["text_full"][:800])

=== PREVIEW TEKS BERSIH (kasus pertama, 800 char) ===
putusan nomor 132/pid. b/2026/pn mlgdemi keadilan berdasarkan ketuhanan yang maha esapengadilan negeri malang yang mengadili perkara pidana denganacara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagaiberikut dalam perkara terdakwa :1.nama lengkap: sendi sulistiyadi 2.tempat lahir: bandung umurtanggal lahir: 39 tahun / 21 september 19864.jenis kelamin: laki-laki ; 5.kebangsaan: indonesia ; 6.tempat tinggal: lapas klas i malang jalan asahan no.7 rt 10/rw 01, kelurahan purwantoro, kec. blimbing, kota malang7.agama: islam ; 8.pekerjaan: belum bekerja ; terdakwa di tahan dalam perkara lain ; terdakwa di persidangan tidak didampingi oleh advokat penasihat hukum; pengadilan negeri tersebut ; setelah membaca :-penetapan ketua pengadilan negeri malang nomor 132/pid. b/2026/pn mlg
